# 05 - Validating the weak labels against a gold sample

The star-derived label is a proxy. To check how well it tracks what reviewers actually write, a stratified sample of 400 substantive reviews (200 Arabic, 200 English, balanced across the two weak labels) is annotated from the text alone, following the rubric in Appendix C. Star rating and weak label are withheld from the annotation sheet.

Agreement is reported as Cohen's kappa, overall and per language, with a bootstrap 95% CI and the Landis and Koch band. A 20% subsample is re-annotated in a separate pass to bound the reliability of the reference itself.

Files:
- `data/annotations/sample_400.csv` - the sheet handed to the annotator (text only)
- `data/annotations/annotations_400.csv` - completed labels (`review_id`, `gold_label`)
- `data/annotations/reannotation_80.csv` - second pass on 80 reviews

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# make src importable when the kernel starts inside notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", context="notebook")
from src import config
from src.io import ensure_dirs, save_table, update_metrics, read_metrics
ensure_dirs()

## Draw the sample (only once; the file is kept so the sample is fixed)

In [2]:
from src.labelling import sample_for_annotation, load_annotations, load_reannotations
from src.stats import cohens_kappa

modelling = pd.read_parquet(config.MODELLING_FILE)
config.ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)

if not config.ANNOTATION_SAMPLE.exists():
    sample = sample_for_annotation(modelling, per_language=200)
    sheet = sample[["annotation_order", "review_id", "language", "review_text"]]
    sheet.to_csv(config.ANNOTATION_SAMPLE, index=False)
    print("annotation sheet written:", len(sheet), "reviews")
sheet = pd.read_csv(config.ANNOTATION_SAMPLE, dtype={"review_id": str})
print(sheet.language.value_counts().to_dict())
sheet.head()

{'English': 200, 'Arabic': 200}


,annotation_order,review_id,language,review_text
0,1,34677f7274f194fd,English,Very Worst App new shail app Compared with Old...
1,2,2fb80d1c18fe4e65,English,It was very bad experience this app very compl...
2,3,cf3056575665f207,Arabic,التطبيق يتوقف نهائيا عند طلب عرض البطاقة ..الح...
3,4,65ac3a0468e6da3c,English,This app is not working or downloading in my m...
4,5,3a3e54149b6fa045,Arabic,التطبيق ممتاز.. ولكن للأسف الزين ما يكمل نواجه...


## Agreement between gold annotation and the weak label

In [3]:
if not config.ANNOTATION_FILE.exists():
    raise SystemExit("annotations_400.csv not found yet - complete the sheet first")

ann = load_annotations()
m = (sheet.merge(ann[["review_id", "gold_label"]], on="review_id", how="left")
          .merge(modelling[["review_id", "satisfaction_label", "star_rating"]], on="review_id"))
print("annotated:", m.gold_label.notna().sum(), "of", len(m))
print(m.gold_label.value_counts(dropna=False).to_dict())

usable = m[m.gold_label.isin(["Satisfied", "Dissatisfied"])]
excluded = len(m) - len(usable)
print(f"excluded as unclear / too short: {excluded}")

rows = []
for name, sub in [("English", usable[usable.language == "English"]),
                  ("Arabic", usable[usable.language == "Arabic"]),
                  ("Overall", usable)]:
    k = cohens_kappa(sub.satisfaction_label, sub.gold_label)
    k["stratum"] = name
    rows.append(k)
kappa = pd.DataFrame(rows)[["stratum", "n", "observed_agreement", "kappa", "ci_low", "ci_high", "band"]]
kappa

annotated: 400 of 400
{'Dissatisfied': 218, 'Satisfied': 167, 'Unclear': 15}
excluded as unclear / too short: 15


,stratum,n,observed_agreement,kappa,ci_low,ci_high,band
0,English,192,0.927083,0.853659,0.778112,0.923722,Almost perfect
1,Arabic,193,0.937824,0.875711,0.803599,0.937802,Almost perfect
2,Overall,385,0.932468,0.864796,0.812887,0.911706,Almost perfect


In [4]:
print("confusion weak label (rows) vs gold (cols):")
print(pd.crosstab(usable.satisfaction_label, usable.gold_label))
print()
print("disagreements by star rating:")
print(pd.crosstab(usable.star_rating, usable.satisfaction_label != usable.gold_label))
print()
print("a few disagreements:")
for _, r in usable[usable.satisfaction_label != usable.gold_label].head(8).iterrows():
    print(f"  [{r.star_rating}*] weak={r.satisfaction_label:12s} gold={r.gold_label:12s} | {r.review_text[:110]}")

confusion weak label (rows) vs gold (cols):
gold_label          Dissatisfied  Satisfied
satisfaction_label                         
Dissatisfied                 193          1
Satisfied                     25        166

disagreements by star rating:
col_0        False  True 
star_rating              
1              173      1
2               20      0
4               14      6
5              152     19

a few disagreements:
  [5*] weak=Satisfied    gold=Dissatisfied | Iqama & Friday khutba. Previous app provided iqama timing centre focused and it was easy to access the Friday 
  [5*] weak=Satisfied    gold=Dissatisfied | Certificate not printing after giving otp, app automatically closes.
  [4*] weak=Satisfied    gold=Dissatisfied | Crashes when I click on next to view more weather info
  [5*] weak=Satisfied    gold=Dissatisfied | Clearance move out card payment does not work
  [5*] weak=Satisfied    gold=Dissatisfied | Im using same password on website that I changed in the App it gi

## Intra-annotator agreement (second pass on 20% of the sample)

In [5]:
if config.REANNOTATION_FILE.exists():
    re = load_reannotations().rename(columns={"gold_label": "gold_label_2"})
    both = ann.merge(re[["review_id", "gold_label_2"]], on="review_id")
    both = both[both.gold_label.isin(["Satisfied", "Dissatisfied"]) & both.gold_label_2.isin(["Satisfied", "Dissatisfied"])]
    intra = cohens_kappa(both.gold_label, both.gold_label_2)
    intra["stratum"] = "Intra-annotator (second pass)"
    kappa = pd.concat([kappa, pd.DataFrame([intra])[kappa.columns]], ignore_index=True)
    print(f"second pass: n={intra['n']}  agreement={intra['observed_agreement']:.3f}  kappa={intra['kappa']:.3f} [{intra['ci_low']:.3f}, {intra['ci_high']:.3f}] {intra['band']}")
else:
    print("no second-pass file yet")
save_table(kappa, "table17_kappa")
update_metrics("kappa", {"rows": kappa.to_dict(orient="records"), "excluded_unclear": int(excluded)})
kappa

second pass: n=76  agreement=0.987  kappa=0.974 [0.919, 1.000] Almost perfect


,stratum,n,observed_agreement,kappa,ci_low,ci_high,band
0,English,192,0.927083,0.853659,0.778112,0.923722,Almost perfect
1,Arabic,193,0.937824,0.875711,0.803599,0.937802,Almost perfect
2,Overall,385,0.932468,0.864796,0.812887,0.911706,Almost perfect
3,Intra-annotator (second pass),76,0.986842,0.973684,0.919492,1.000000,Almost perfect
